In [1]:
"""
stage_5.py -- Stage 5 (iteration 2): 5c2-raw hazard model, manifest-driven.

Frozen model contract (5c2-raw): grammar (bsince + ewm{2,6,18} per class, age, tod)
+ causal z-scored values @ {t-1, t-2, t-3}: signed & magnitude of every stream's
source column, leg amplitude, raw body signed & magnitude. Sign convention
confirming-positive (value * leg_dir).

Manifest-driven (iteration 2):
  - Fork axes (frame, session window, stream set) are READ from the Stage-0
    manifest, never redeclared here. Dropping a stream in Stage 0 (e.g. no-TICK)
    propagates automatically: grammar classes AND z-value channels both track the
    manifest stream set.
  - tod is derived from clock time (session_start), resolution-independent.
  - The manifest is baked into the model bundle so the booster carries its own
    contract; the worker asserts against it and prints it at startup.

Naming (iteration 2):
  SOURCE_PATH / src : the "source" oscillator file (HA OHLC + JMA + TICK + derivs),
                      Stage-0's input; rawer than bars/events. src is the primary
                      data frame, augmented in-place with raw body columns.
  RAW_FILE / raw1   : pure MNQ raw OHLC (transient; merged into src, then unused).
"""

import json
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from scipy.signal import lfilter
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, roc_auc_score

from common import Featurizer, load_manifest, _expanding_z, _welford_check

In [2]:
# ---------------------------------------------------------------- CONFIG (per-run, explicit)
FRAME = 3
STAGE0_TAG = 'mnq-4STREAM-ALL-9-12am'

MANIFEST_PATH = f"stage-0/manifest_{STAGE0_TAG}_{FRAME}s.json"
BARS_PATH = f"stage-0/bars_{STAGE0_TAG}_{FRAME}s.pqt"
EVENTS_PATH = f"stage-0/events_{STAGE0_TAG}_{FRAME}s.pqt"

ITER_DIR = "."                                   # iteration-2 root (encapsulated)
OUT_DIR = "stage-5"

VALID_FROM = "2025-07-01"
TRAIN_END = "2025-12-31"
TEST_FROM = "2026-01-01"

# frozen 5c2 architecture constants (NOT fork axes -- stay in code)
TAUS = (2.0, 6.0, 18.0)
VALUE_LAGS = (1, 2)                               # t-2, t-3 (t-1 shift is implicit)
ZWARM = 20
TOD_BIN_MIN = 30
BODY_TAG = "raw"
BODY_OPEN_COL = f"{BODY_TAG}Open"
BODY_CLOSE_COL = f"{BODY_TAG}Last"

LGBM_PARAMS = dict(
    objective="binary", metric="binary_logloss", learning_rate=0.05,
    num_leaves=127, min_data_in_leaf=1000, feature_fraction=0.9,
    bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
    num_threads=28, verbosity=-1
)
NUM_ROUNDS = 8000
EARLY_STOP = 200
STAGE5_ANCHOR = 0.27402
# ----------------------------------------------------------------

In [3]:
# ---------------------------------------------------------------- grammar features
def build_grammar_features(fz, date_from=None, date_to=None):
    blocks = []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        n = S["n"]
        cols = []
        for c in fz.classes:
            P = S["P"][c]
            ind = np.diff(P).astype(np.float64)
            occ = np.where(ind > 0, np.arange(n), -1)
            last = np.maximum.accumulate(occ)
            lastm1 = np.concatenate(([-1], last[:-1]))
            bsince = np.where(lastm1 >= 0, np.arange(n) - lastm1, np.arange(n) + 1)
            cols.append(bsince[t])
            x = np.concatenate(([0.0], ind[:-1]))
            for tau in TAUS:
                a = np.exp(-1.0 / tau)
                s = lfilter([a], [1.0, -a], x)
                cols.append(s[t])
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)
        age = np.where(lt >= 0, t - lt, t + 1)
        cols.append(age)
        cols.append(S["tod"][t].astype(np.float64))
        blocks.append(np.stack(cols, 1).astype(np.float32))
    return np.concatenate(blocks), fz.grammar_names

In [4]:
# ---------------------------------------------------------------- value features
def value_base_names(manifest):
    """Value channels derived from the manifest stream set (source columns)."""
    cols = manifest["_stream_cols"]
    names = ([f"z_{c}_signed" for c in cols]
             + [f"z_{c}_mag" for c in cols]
             + ["z_leg_amp", f"z_body_{BODY_TAG}_signed", f"z_body_{BODY_TAG}_mag"])
    return cols, names


def build_value_features(fz, src, date_from=None, date_to=None):
    cols, base = value_base_names(fz.manifest)
    names = base + [f"{nm}_lag{L}" for L in VALUE_LAGS for nm in base]
    sv = src.set_index("timestamp")
    blocks = []
    for S in fz._selected(date_from, date_to):
        ts = pd.DatetimeIndex(S["timestamp"])
        r = sv.reindex(ts)
        leg_dir = S["leg_dir"]
        feats = []
        for c in cols:                                             # signed per stream col
            feats.append(_expanding_z(r[c].to_numpy(np.float64) * leg_dir, ZWARM))
        for c in cols:                                             # magnitude per stream col
            feats.append(_expanding_z(np.abs(r[c].to_numpy(np.float64)), ZWARM))
        feats.append(_expanding_z(np.abs(r["JMA"].to_numpy(np.float64) - S["leg_start_jma"]), ZWARM))
        bo = r[BODY_OPEN_COL].to_numpy(np.float64)
        bc = r[BODY_CLOSE_COL].to_numpy(np.float64)
        feats.append(_expanding_z((bc - bo) * leg_dir, ZWARM))     # confirming-positive
        feats.append(_expanding_z(np.abs(bc - bo), ZWARM))

        M = np.stack(feats, 1)
        M = np.concatenate([np.zeros((1, M.shape[1])), M[:-1]], 0)          # t-1 shift
        lagged = [M]
        for L in VALUE_LAGS:
            lagged.append(np.concatenate([np.zeros((L, M.shape[1])), M[:-L]], 0))
        M = np.concatenate(lagged, 1)
        t = np.nonzero(~S["warm"])[0]
        Mt = M[t]
        Mt[(t < ZWARM + max(VALUE_LAGS))] = 0.0
        blocks.append(Mt.astype(np.float32))
    return np.concatenate(blocks), names


def build_X(fz, src, date_from=None, date_to=None):
    Xg, gn = build_grammar_features(fz, date_from, date_to)
    Xv, vn = build_value_features(fz, src, date_from, date_to)
    assert len(Xg) == len(Xv), (len(Xg), len(Xv))
    return np.hstack([Xg, Xv]), gn + vn


def build_meta(fz, date_from=None, date_to=None):
    bi, ts, tg, dt = [], [], [], []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        bi.append(S["bar_index"][t])
        ts.append(S["timestamp"][t])
        tg.append(S["tgt"][t])
        dt.append(np.full(len(t), str(S["sess"])))
    return pd.DataFrame({"bar_index": np.concatenate(bi),
                         "timestamp": np.concatenate(ts),
                         "is_target": np.concatenate(tg),
                         "date": np.concatenate(dt)})

In [5]:
# ---------------------------------------------------------------- train / eval
def train(fz, src, train_end, valid_from):
    X, names = build_X(fz, src, None, train_end)
    meta = build_meta(fz, None, train_end)
    y = meta["is_target"].to_numpy().astype(np.int8)
    va = (meta["date"] >= valid_from).to_numpy()
    tr = ~va
    dtr = lgb.Dataset(X[tr], label=y[tr], feature_name=names)
    dva = lgb.Dataset(X[va], label=y[va], reference=dtr)
    booster = lgb.train(LGBM_PARAMS, dtr, num_boost_round=NUM_ROUNDS,
                        valid_sets=[dva], valid_names=["valid"],
                        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False),
                                   lgb.log_evaluation(200)])
    p_va = booster.predict(X[va], num_iteration=booster.best_iteration)
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_va, y[va])
    print(json.dumps(dict(n_train=int(tr.sum()), n_valid=int(va.sum()),
                          best_iteration=int(booster.best_iteration),
                          valid_logloss_cal=float(log_loss(y[va], iso.predict(p_va)))),
                     indent=2))
    imp = pd.DataFrame({"feature": names,
                        "gain": booster.feature_importance("gain")}
                       ).sort_values("gain", ascending=False)
    print(imp.to_string(index=False))
    return dict(booster=booster, iso=iso, feature_names=names,
                valid_from=valid_from, train_end=train_end,
                manifest=fz.manifest, tag=STAGE0_TAG, importance=imp)


def evaluate(fz, src, model, start, end=None, anchor=STAGE5_ANCHOR):
    X, _ = build_X(fz, src, start, end)
    meta = build_meta(fz, start, end)
    y = meta["is_target"].to_numpy().astype(np.int8)
    p = model["booster"].predict(X, num_iteration=model["booster"].best_iteration)
    p_cal = model["iso"].predict(p)
    ll_cal = log_loss(y, p_cal)
    ll_const = log_loss(y, np.full_like(p, y.mean(), dtype=np.float64))
    print(json.dumps(dict(n_rows=int(len(y)), holdout_logloss_cal=float(ll_cal),
                          holdout_logloss_const=float(ll_const),
                          skill=float(1 - ll_cal / ll_const),
                          auc=float(roc_auc_score(y, p)),
                          anchor=anchor, delta=float(ll_cal - anchor)), indent=2))
    out = meta[["bar_index", "timestamp", "is_target"]].copy()
    out["p"] = p.astype(np.float32)
    out["p_cal"] = p_cal.astype(np.float32)
    tbl = out.assign(bin=pd.qcut(out["p_cal"], 10, duplicates="drop")).groupby(
        "bin", observed=True).agg(mean_p=("p_cal", "mean"),
                                  realized=("is_target", "mean"), n=("p_cal", "size"))
    print(tbl.to_string())
    return out

In [6]:
# ---------------------------------------------------------------- run
manifest = load_manifest(MANIFEST_PATH,TOD_BIN_MIN)

SOURCE_PATH = manifest["source_file"]

assert STAGE0_TAG == manifest["stage0_tag"]

bars = pd.read_parquet(BARS_PATH)
events = pd.read_parquet(EVENTS_PATH)

sess_lo = pd.Timestamp(manifest["session_start"]).time()
sess_hi = pd.Timestamp(manifest["session_end"]).time()

src = pd.read_parquet(SOURCE_PATH)
src = src[(src["timestamp"].dt.time >= sess_lo) & (src["timestamp"].dt.time < sess_hi)]


################################################
# REMOVE ME - trying to align with previous runs
################################################
src = src[src['date'] <= pd.to_datetime('2026-07-08')]
    

# assert day start exists in src and raw ###
src_min_time = src["timestamp"].dt.time.min()
print(f'SRC MIN TIME: {src_min_time}')
assert sess_lo == src_min_time

assert src[["rawOpen", "rawLast"]].notna().all().all(), "raw OHLC has gaps vs source timestamps"

SRC MIN TIME: 09:00:00


In [7]:
print('----------------------------- !! VERIFY !! -----------------------------')
print(f'FRAME: {FRAME}sec, STAGE0_TAG: {STAGE0_TAG}, BODY_TAG: {BODY_TAG}')
print('----------------------- MANIFEST -----------------------')
print(json.dumps({k: v for k, v in manifest.items() if not k.startswith("_")}, indent=2))
print('------------------------------------------------------------------------')

#

fz = Featurizer(bars, events, manifest, TOD_BIN_MIN, TAUS)
#augment_featurizer(fz, bars)

S0 = fz.sessions[len(fz.sessions) // 2]
xchk = src.set_index("timestamp").reindex(pd.DatetimeIndex(S0["timestamp"]))["jmaD1"].to_numpy(np.float64)
print("welford max abs diff:", _welford_check(xchk, ZWARM))

model = train(fz, src, TRAIN_END, VALID_FROM)
pred = evaluate(fz, src, model, TEST_FROM)

print(f"-------------- {STAGE0_TAG} {FRAME}s --------------")
joblib_file = f"{OUT_DIR}/model_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.joblib"
importance_file = f"{OUT_DIR}/importance_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.csv"
pred_file = f"{OUT_DIR}/pred_{BODY_TAG}_{STAGE0_TAG}_{FRAME}s.pqt"

joblib.dump({k: v for k, v in model.items() if k != "importance"}, joblib_file)
model["importance"].to_csv(importance_file, index=False)
pred.to_parquet(pred_file, index=False)

print(f'    joblib_file: {joblib_file}')
print(f'importance_file: {importance_file}')
print(f'      pred_file: {pred_file}')

#

z = pred.timestamp >= TEST_FROM
print("holdout-window logloss:", log_loss(pred.is_target[z], pred.p_cal[z]))

#

print('\n--------------------------- SUMMARY ---------------------------\n')

p_date = pred['timestamp'].dt.normalize()

h = pred[p_date >= "2026-01-01"]          # no-op if the parquet is holdout-only
GREEN = float(h["p_cal"].quantile(0.50))
RED   = float(h["p_cal"].quantile(0.90))
g = h[h["p_cal"] <  GREEN]
r = h[h["p_cal"] >= RED]
print(f"GREEN {GREEN:.8g}  {len(g)/len(h):.2%} of bars  wrong 1/{1/g['is_target'].mean():.0f}")
print(f"RED   {RED:.8g}    {len(r)/len(h):.2%} of bars  right {r['is_target'].mean():.1%}")

#

y = h["is_target"].values.astype(np.float64)
pc = np.clip(h["p_cal"].values.astype(np.float64), 1e-15, 1-1e-15)
a  = y.mean()
ll_model = -(y*np.log(pc) + (1-y)*np.log(1-pc)).mean()
ll_const = -(a*np.log(a) + (1-a)*np.log(1-a))
print(f"y.mean {a:.5f}  ll_model {ll_model:.5f}  ll_const {ll_const:.5f}  skill {1-ll_model/ll_const:.4f}  rows {len(h)}")

----------------------------- !! VERIFY !! -----------------------------
FRAME: 3sec, STAGE0_TAG: mnq-3STREAM-ALL-X-9-12am, BODY_TAG: raw
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-all-3sec.pqt",
  "n_bars": 4120088,
  "n_sessions": 1145,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-08 00:00:00",
  "created": "2026-07-26T17:45:46.279360",
  "stage0_tag": "mnq-3STREAM-ALL-X-9-12am"
}
------------------------------------------------------------------------
welford max abs diff: 4.440892098500626e-15
[200]	valid's binary_logloss: 0.133711
[400]	valid's binary_logloss: 0.129995
[600]	valid's binar

In [8]:
'''
--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: mnq-TICK-RSX-9-12am, BODY_TAG: raw
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    },
    {
      "name": "TICK_D2",
      "column": "tickJmaD2"
    },
    {
      "name": "MNQ_RSX",
      "column": "RSX"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 4120088,
  "n_sessions": 1145,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-08 00:00:00",
  "created": "2026-07-26T13:07:46.967177",
  "stage0_tag": "mnq-TICK-RSX-9-12am"
}
--------------------------------------------------------
welford max abs diff: 4.440892098500626e-15
[200]	valid's binary_logloss: 0.133554
[400]	valid's binary_logloss: 0.129208
[600]	valid's binary_logloss: 0.127996
[800]	valid's binary_logloss: 0.127516
[1000]	valid's binary_logloss: 0.12723
[1200]	valid's binary_logloss: 0.127097
[1400]	valid's binary_logloss: 0.126973
[1600]	valid's binary_logloss: 0.12691
[1800]	valid's binary_logloss: 0.126856
[2000]	valid's binary_logloss: 0.126853
[2200]	valid's binary_logloss: 0.12685
{
  "n_train": 3175513,
  "n_valid": 452137,
  "best_iteration": 2141,
  "valid_logloss_cal": 0.12631677035498085
}
                feature         gain
      z_body_raw_signed 2.095796e+06
         z_jmaD2_signed 1.342334e+06
            z_jmaD1_mag 1.297724e+06
         z_jmaD1_signed 1.142142e+06
 z_body_raw_signed_lag1 1.102208e+06
         z_body_raw_mag 9.089110e+05
       z_jmaD1_mag_lag1 5.665001e+05
    z_body_raw_mag_lag1 3.279649e+05
    z_jmaD1_signed_lag1 2.954392e+05
     MNQ_D1|conf|bsince 2.668906e+05
 z_body_raw_signed_lag2 2.599357e+05
      MNQ_D1|opp|bsince 2.524063e+05
            z_jmaD2_mag 2.222439e+05
MNQ_JMA_SELF|all|bsince 2.003550e+05
        MNQ_D1|opp|ewm2 1.937338e+05
  MNQ_JMA_SELF|all|ewm2 1.402856e+05
    z_jmaD1_signed_lag2 1.346284e+05
       z_jmaD1_mag_lag2 1.292727e+05
                    tod 1.165696e+05
       MNQ_D1|conf|ewm2 1.139449e+05
{
  "n_rows": 477398,
  "holdout_logloss_cal": 0.1231558773825773,
  "holdout_logloss_const": 0.3334844163241089,
  "skill": 0.6306997528097871,
  "auc": 0.97154778473756,
  "anchor": 0.27402,
  "delta": -0.15086412261742269
}
                        mean_p  realized      n
bin                                            
(-0.001, 4.17e-05]    0.000034  0.000084  59244
(4.17e-05, 0.000206]  0.000206  0.000312  44804
(0.000206, 0.00045]   0.000438  0.000615  55263
(0.00045, 0.000932]   0.000730  0.000968  38240
(0.000932, 0.00213]   0.001600  0.001395  43735
(0.00213, 0.00613]    0.004366  0.003239  48466
(0.00613, 0.0189]     0.012731  0.012284  45019
(0.0189, 0.0986]      0.053848  0.052015  52389
(0.0986, 0.391]       0.231949  0.233226  43640
(0.391, 1.0]          0.745474  0.768681  46598
-------------- mnq-TICK-RSX-9-12am 3s --------------
    joblib_file: stage-5/model_raw_mnq-TICK-RSX-9-12am_3s.joblib
importance_file: stage-5/importance_raw_mnq-TICK-RSX-9-12am_3s.csv
      pred_file: stage-5/pred_raw_mnq-TICK-RSX-9-12am_3s.pqt
holdout-window logloss: 0.12302955985069275

==================================================================================
================================== SUMMARY =======================================
==================================================================================
stage-5/pred_raw_mnq-TICK-RSX-9-12am_3s.pqt

GREEN 0.0021303792  48.38% of bars  wrong 1/1686
RED   0.39110252    10.31% of bars  right 75.0%
y.mean 0.10386  ll_model 0.12315  ll_const 0.33348  skill 0.6307  rows 477398
'''

'\n--------------------- !! VERIFY !! ---------------------\nFRAME: 3sec, STAGE0_TAG: mnq-TICK-RSX-9-12am, BODY_TAG: raw\n----------------------- MANIFEST -----------------------\n{\n  "frame_seconds": 3,\n  "session_start": "09:00",\n  "session_end": "12:00",\n  "warmup_bars": 10,\n  "streams": [\n    {\n      "name": "MNQ_D1",\n      "column": "jmaD1"\n    },\n    {\n      "name": "MNQ_D2",\n      "column": "jmaD2"\n    },\n    {\n      "name": "TICK_D1",\n      "column": "tickJmaD1"\n    },\n    {\n      "name": "TICK_D2",\n      "column": "tickJmaD2"\n    },\n    {\n      "name": "MNQ_RSX",\n      "column": "RSX"\n    }\n  ],\n  "self_stream": "MNQ_JMA_SELF",\n  "source_file": "data/mnq-tick-full-3sec.pqt",\n  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",\n  "n_bars": 4120088,\n  "n_sessions": 1145,\n  "date_min": "2022-01-03 00:00:00",\n  "date_max": "2026-07-08 00:00:00",\n  "created": "2026-07-26T13:07:46.967177",\n  "stage0_tag": "mnq-TICK-RSX-9-12am"\n}\n--------------------------

In [9]:
''' mnq-3STREAM-ALL-9-12am  FAIL
----------------------------- !! VERIFY !! -----------------------------
FRAME: 3sec, STAGE0_TAG: mnq-3STREAM-ALL-9-12am, BODY_TAG: raw
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-all-3sec.pqt",
  "n_bars": 4123685,
  "n_sessions": 1146,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-10 00:00:00",
  "created": "2026-07-26T16:48:08.134112",
  "stage0_tag": "mnq-3STREAM-ALL-9-12am"
}
------------------------------------------------------------------------
welford max abs diff: 4.440892098500626e-15
[200]	valid's binary_logloss: 0.133711
[400]	valid's binary_logloss: 0.129995
[600]	valid's binary_logloss: 0.129006
[800]	valid's binary_logloss: 0.128591
[1000]	valid's binary_logloss: 0.128442
[1200]	valid's binary_logloss: 0.128376
[1400]	valid's binary_logloss: 0.128276
[1600]	valid's binary_logloss: 0.128252
{
  "n_train": 3175513,
  "n_valid": 452137,
  "best_iteration": 1571,
  "valid_logloss_cal": 0.1277844719216301
}
                feature         gain
      z_body_raw_signed 2.050344e+06
            z_jmaD1_mag 1.557306e+06
         z_jmaD2_signed 1.348822e+06
 z_body_raw_signed_lag1 1.061368e+06
         z_jmaD1_signed 9.708696e+05
         z_body_raw_mag 9.346517e+05
       z_jmaD1_mag_lag1 5.711367e+05
    z_jmaD1_signed_lag1 3.528163e+05
    z_body_raw_mag_lag1 3.136217e+05
     MNQ_D1|conf|bsince 3.015361e+05
      MNQ_D1|opp|bsince 2.654141e+05
 z_body_raw_signed_lag2 2.471321e+05
            z_jmaD2_mag 2.048486e+05
        MNQ_D1|opp|ewm2 1.708272e+05
MNQ_JMA_SELF|all|bsince 1.428747e+05
  MNQ_JMA_SELF|all|ewm2 1.428052e+05
       z_jmaD1_mag_lag2 1.398708e+05
                    tod 1.131920e+05
    z_jmaD1_signed_lag2 1.032075e+05
    z_jmaD2_signed_lag1 9.501618e+04
       MNQ_D1|conf|ewm2 9.265089e+04
       z_jmaD2_mag_lag1 8.619701e+04
    z_body_raw_mag_lag2 6.257742e+04
       z_jmaD2_mag_lag2 5.922899e+04
        MNQ_D1|opp|ewm6 5.516455e+04
    z_jmaD2_signed_lag2 4.498036e+04
        MNQ_D2|opp|ewm2 4.238488e+04
 MNQ_JMA_SELF|all|ewm18 4.223020e+04
         z_leg_amp_lag1 3.851364e+04
         z_leg_amp_lag2 3.789704e+04
              z_leg_amp 3.749510e+04
        z_tickJmaD1_mag 3.564404e+04
       MNQ_D2|conf|ewm2 3.496647e+04
     TICK_D1|conf|ewm18 3.453802e+04
      MNQ_D1|conf|ewm18 3.341752e+04
       MNQ_D1|opp|ewm18 3.321388e+04
      TICK_D1|opp|ewm18 3.250532e+04
       MNQ_D2|opp|ewm18 3.171442e+04
      MNQ_D2|conf|ewm18 3.170095e+04
     z_tickJmaD1_signed 3.068924e+04
   z_tickJmaD1_mag_lag2 3.006224e+04
                    age 2.924590e+04
        MNQ_D2|opp|ewm6 2.706514e+04
   z_tickJmaD1_mag_lag1 2.695670e+04
  MNQ_JMA_SELF|all|ewm6 2.644090e+04
z_tickJmaD1_signed_lag2 2.636491e+04
z_tickJmaD1_signed_lag1 2.605653e+04
       TICK_D1|opp|ewm2 2.505436e+04
       TICK_D1|opp|ewm6 2.375975e+04
       MNQ_D1|conf|ewm6 2.241835e+04
      TICK_D1|conf|ewm2 2.239777e+04
       MNQ_D2|conf|ewm6 2.223355e+04
      TICK_D1|conf|ewm6 2.132544e+04
     MNQ_D2|conf|bsince 8.489798e+03
    TICK_D1|conf|bsince 5.947509e+03
     TICK_D1|opp|bsince 4.401485e+03
      MNQ_D2|opp|bsince 2.946320e+03
{
  "n_rows": 480985,
  "holdout_logloss_cal": 0.12460844907490194,
  "holdout_logloss_const": 0.33351779838862367,
  "skill": 0.6263814114960518,
  "auc": 0.9705719031364917,
  "anchor": 0.27402,
  "delta": -0.14941155092509806
}
                        mean_p  realized      n
bin                                            
(-0.001, 6.79e-05]    0.000058  0.000227  52877
(6.79e-05, 0.000373]  0.000289  0.000307  68404
(0.000373, 0.000474]  0.000470  0.000807  35933
(0.000474, 0.00152]   0.001124  0.001044  55570
(0.00152, 0.00284]    0.002152  0.001779  42165
(0.00284, 0.00765]    0.005773  0.005235  43938
(0.00765, 0.021]      0.014584  0.013172  37883
(0.021, 0.0914]       0.051843  0.050122  49958
(0.0914, 0.384]       0.215725  0.219167  46412
(0.384, 1.0]          0.735381  0.760017  47845
-------------- mnq-3STREAM-ALL-9-12am 3s --------------
    joblib_file: stage-5/model_raw_mnq-3STREAM-ALL-9-12am_3s.joblib
importance_file: stage-5/importance_raw_mnq-3STREAM-ALL-9-12am_3s.csv
      pred_file: stage-5/pred_raw_mnq-3STREAM-ALL-9-12am_3s.pqt
holdout-window logloss: 0.12456665933132172

--------------------------- SUMMARY ---------------------------

GREEN 0.0028354081  49.74% of bars  wrong 1/1441
RED   0.38397247    10.42% of bars  right 74.3%
y.mean 0.10388  ll_model 0.12461  ll_const 0.33352  skill 0.6264  rows 480985
'''

' mnq-3STREAM-ALL-9-12am  FAIL\n----------------------------- !! VERIFY !! -----------------------------\nFRAME: 3sec, STAGE0_TAG: mnq-3STREAM-ALL-9-12am, BODY_TAG: raw\n----------------------- MANIFEST -----------------------\n{\n  "frame_seconds": 3,\n  "session_start": "09:00",\n  "session_end": "12:00",\n  "warmup_bars": 10,\n  "streams": [\n    {\n      "name": "MNQ_D1",\n      "column": "jmaD1"\n    },\n    {\n      "name": "MNQ_D2",\n      "column": "jmaD2"\n    },\n    {\n      "name": "TICK_D1",\n      "column": "tickJmaD1"\n    }\n  ],\n  "self_stream": "MNQ_JMA_SELF",\n  "source_file": "data/mnq-tick-all-3sec.pqt",\n  "n_bars": 4123685,\n  "n_sessions": 1146,\n  "date_min": "2022-01-03 00:00:00",\n  "date_max": "2026-07-10 00:00:00",\n  "created": "2026-07-26T16:48:08.134112",\n  "stage0_tag": "mnq-3STREAM-ALL-9-12am"\n}\n------------------------------------------------------------------------\nwelford max abs diff: 4.440892098500626e-15\n[200]\tvalid\'s binary_logloss: 0.13

In [10]:
'''
                feature         gain
      z_body_raw_signed 2.065041e+06
            z_jmaD1_mag 1.568559e+06
         z_jmaD2_signed 1.353643e+06
 z_body_raw_signed_lag1 1.072868e+06
         z_jmaD1_signed 9.638962e+05
         z_body_raw_mag 9.365891e+05
       z_jmaD1_mag_lag1 5.744333e+05
    z_jmaD1_signed_lag1 3.500731e+05
    z_body_raw_mag_lag1 3.197378e+05
     MNQ_D1|conf|bsince 3.013779e+05
      MNQ_D1|opp|bsince 2.679680e+05
 z_body_raw_signed_lag2 2.551477e+05
            z_jmaD2_mag 2.100125e+05
        MNQ_D1|opp|ewm2 1.775525e+05
  MNQ_JMA_SELF|all|ewm2 1.499033e+05
MNQ_JMA_SELF|all|bsince 1.456360e+05
       z_jmaD1_mag_lag2 1.436053e+05
                    tod 1.216548e+05
    z_jmaD1_signed_lag2 1.064752e+05
       MNQ_D1|conf|ewm2 9.926765e+04
{
  "n_rows": 477398,
  "holdout_logloss_cal": 0.12296254907057177,
  "holdout_logloss_const": 0.3334844163241089,
  "skill": 0.6312794749873225,
  "auc": 0.9716153733084922,
  "anchor": 0.27402,
  "delta": -0.1510574509294282

}
                        mean_p  realized      n
bin                                            
(-0.001, 7.76e-05]    0.000040  0.000118  50796
(7.76e-05, 0.000227]  0.000212  0.000207  62916
(0.000227, 0.000547]  0.000409  0.000542  33213
(0.000547, 0.000962]  0.000824  0.000938  57551
(0.000962, 0.00289]   0.002217  0.001740  55763
(0.00289, 0.0059]     0.004754  0.004150  26266
(0.0059, 0.0194]      0.011943  0.011119  48294
(0.0194, 0.0889]      0.051969  0.049666  49531
(0.0889, 0.4]         0.226288  0.229589  47463
(0.4, 1.0]            0.753017  0.776055  45605
-------------- mnq-3STREAM-9-12am 3s --------------
    joblib_file: stage-5/model_raw_mnq-3STREAM-9-12am_3s.joblib
importance_file: stage-5/importance_raw_mnq-3STREAM-9-12am_3s.csv
      pred_file: stage-5/pred_raw_mnq-3STREAM-9-12am_3s.pqt
holdout-window logloss: 0.12287834286689758
'''

'\n                feature         gain\n      z_body_raw_signed 2.065041e+06\n            z_jmaD1_mag 1.568559e+06\n         z_jmaD2_signed 1.353643e+06\n z_body_raw_signed_lag1 1.072868e+06\n         z_jmaD1_signed 9.638962e+05\n         z_body_raw_mag 9.365891e+05\n       z_jmaD1_mag_lag1 5.744333e+05\n    z_jmaD1_signed_lag1 3.500731e+05\n    z_body_raw_mag_lag1 3.197378e+05\n     MNQ_D1|conf|bsince 3.013779e+05\n      MNQ_D1|opp|bsince 2.679680e+05\n z_body_raw_signed_lag2 2.551477e+05\n            z_jmaD2_mag 2.100125e+05\n        MNQ_D1|opp|ewm2 1.775525e+05\n  MNQ_JMA_SELF|all|ewm2 1.499033e+05\nMNQ_JMA_SELF|all|bsince 1.456360e+05\n       z_jmaD1_mag_lag2 1.436053e+05\n                    tod 1.216548e+05\n    z_jmaD1_signed_lag2 1.064752e+05\n       MNQ_D1|conf|ewm2 9.926765e+04\n{\n  "n_rows": 477398,\n  "holdout_logloss_cal": 0.12296254907057177,\n  "holdout_logloss_const": 0.3334844163241089,\n  "skill": 0.6312794749873225,\n  "auc": 0.9716153733084922,\n  "anchor": 0.2740

In [11]:
'''
--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: mnq-VEL-9-12am, BODY_TAG: raw
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    },
    {
      "name": "MNQ_VEL",
      "column": "VEL"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 4120088,
  "n_sessions": 1145,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-08 00:00:00",
  "created": "2026-07-26T13:52:35.026471",
  "stage0_tag": "mnq-VEL-9-12am"
}
--------------------------------------------------------
welford max abs diff: 4.440892098500626e-15
[200]	valid's binary_logloss: 0.133193
[400]	valid's binary_logloss: 0.128895
[600]	valid's binary_logloss: 0.127709
[800]	valid's binary_logloss: 0.127237
[1000]	valid's binary_logloss: 0.126991
[1200]	valid's binary_logloss: 0.126863
[1400]	valid's binary_logloss: 0.12674
[1600]	valid's binary_logloss: 0.126717
[1800]	valid's binary_logloss: 0.126717
{
  "n_train": 3175513,
  "n_valid": 452137,
  "best_iteration": 1662,
  "valid_logloss_cal": 0.12619328428740642
}
                feature         gain
      z_body_raw_signed 2.033859e+06
            z_jmaD1_mag 1.399423e+06
         z_jmaD2_signed 1.298214e+06
         z_jmaD1_signed 1.074316e+06
 z_body_raw_signed_lag1 1.070665e+06
         z_body_raw_mag 9.683603e+05
       z_jmaD1_mag_lag1 5.152271e+05
    z_jmaD1_signed_lag1 3.369553e+05
    z_body_raw_mag_lag1 3.242198e+05
      MNQ_D1|opp|bsince 2.981497e+05
     MNQ_D1|conf|bsince 2.776131e+05
 z_body_raw_signed_lag2 2.588488e+05
MNQ_JMA_SELF|all|bsince 2.392064e+05
            z_jmaD2_mag 2.224568e+05
       z_jmaD1_mag_lag2 1.853623e+05
        MNQ_D1|opp|ewm2 1.506746e+05
                    tod 1.198626e+05
  MNQ_JMA_SELF|all|ewm2 1.151418e+05
    z_jmaD1_signed_lag2 1.113593e+05
    z_jmaD2_signed_lag1 9.193264e+04
{
  "n_rows": 477398,
  "holdout_logloss_cal": 0.12294954519143493,
  "holdout_logloss_const": 0.3334844163241089,
  "skill": 0.6313184689507592,
  "auc": 0.9716845393094683,
  "anchor": 0.27402,
  "delta": -0.15107045480856507
}
                        mean_p  realized      n
bin                                            
(-0.001, 0.000102]    0.000073  0.000132  83601
(0.000102, 0.000316]  0.000316  0.000404  69259
(0.000316, 0.000885]  0.000805  0.000892  60540
(0.000885, 0.00216]   0.001868  0.001444  25617
(0.00216, 0.00552]    0.004145  0.003388  47817
(0.00552, 0.0239]     0.012369  0.011425  47789
(0.0239, 0.092]       0.050922  0.049427  49426
(0.092, 0.376]        0.217517  0.220746  45917
(0.376, 1.0]          0.740402  0.762481  47432
-------------- mnq-VEL-9-12am 3s --------------
    joblib_file: stage-5/model_raw_mnq-VEL-9-12am_3s.joblib
importance_file: stage-5/importance_raw_mnq-VEL-9-12am_3s.csv
      pred_file: stage-5/pred_raw_mnq-VEL-9-12am_3s.pqt
holdout-window logloss: 0.12282324582338333

--------------------------- SUMMARY ---------------------------

GREEN 0.002155351  47.31% of bars  wrong 1/2035
RED   0.37635869    10.10% of bars  right 75.7%
y.mean 0.10386  ll_model 0.12294  ll_const 0.33348  skill 0.6313  rows 477398
'''

'\n--------------------- !! VERIFY !! ---------------------\nFRAME: 3sec, STAGE0_TAG: mnq-VEL-9-12am, BODY_TAG: raw\n----------------------- MANIFEST -----------------------\n{\n  "frame_seconds": 3,\n  "session_start": "09:00",\n  "session_end": "12:00",\n  "warmup_bars": 10,\n  "streams": [\n    {\n      "name": "MNQ_D1",\n      "column": "jmaD1"\n    },\n    {\n      "name": "MNQ_D2",\n      "column": "jmaD2"\n    },\n    {\n      "name": "TICK_D1",\n      "column": "tickJmaD1"\n    },\n    {\n      "name": "MNQ_VEL",\n      "column": "VEL"\n    }\n  ],\n  "self_stream": "MNQ_JMA_SELF",\n  "source_file": "data/mnq-tick-full-3sec.pqt",\n  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",\n  "n_bars": 4120088,\n  "n_sessions": 1145,\n  "date_min": "2022-01-03 00:00:00",\n  "date_max": "2026-07-08 00:00:00",\n  "created": "2026-07-26T13:52:35.026471",\n  "stage0_tag": "mnq-VEL-9-12am"\n}\n--------------------------------------------------------\nwelford max abs diff: 4.440892098500626e-15\n[20

In [12]:
'''
--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: mnq-3STREAM-VEL-9-12am, BODY_TAG: raw
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "MNQ_VEL",
      "column": "VEL"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 4120088,
  "n_sessions": 1145,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-08 00:00:00",
  "created": "2026-07-26T14:11:03.993239",
  "stage0_tag": "mnq-3STREAM-VEL-9-12am"
}
--------------------------------------------------------
welford max abs diff: 4.440892098500626e-15
[200]	valid's binary_logloss: 0.133575
[400]	valid's binary_logloss: 0.129773
[600]	valid's binary_logloss: 0.128708
[800]	valid's binary_logloss: 0.128276
[1000]	valid's binary_logloss: 0.128054
[1200]	valid's binary_logloss: 0.127958
[1400]	valid's binary_logloss: 0.127874
[1600]	valid's binary_logloss: 0.127835
[1800]	valid's binary_logloss: 0.12784
{
  "n_train": 3175513,
  "n_valid": 452137,
  "best_iteration": 1639,
  "valid_logloss_cal": 0.12736450465171623
}
                feature         gain
      z_body_raw_signed 2.054280e+06
            z_jmaD1_mag 1.544580e+06
         z_jmaD2_signed 1.351573e+06
 z_body_raw_signed_lag1 1.063664e+06
         z_jmaD1_signed 9.838435e+05
         z_body_raw_mag 9.346615e+05
       z_jmaD1_mag_lag1 5.782467e+05
    z_jmaD1_signed_lag1 3.477312e+05
    z_body_raw_mag_lag1 3.171422e+05
     MNQ_D1|conf|bsince 3.038103e+05
      MNQ_D1|opp|bsince 2.646146e+05
 z_body_raw_signed_lag2 2.528633e+05
            z_jmaD2_mag 2.048442e+05
        MNQ_D1|opp|ewm2 1.740016e+05
       z_jmaD1_mag_lag2 1.452451e+05
  MNQ_JMA_SELF|all|ewm2 1.447251e+05
MNQ_JMA_SELF|all|bsince 1.432376e+05
                    tod 1.178473e+05
    z_jmaD1_signed_lag2 1.051663e+05
    z_jmaD2_signed_lag1 9.758493e+04
{
  "n_rows": 477398,
  "holdout_logloss_cal": 0.12443772961217549,
  "holdout_logloss_const": 0.3334844163241089,
  "skill": 0.6268559383259571,
  "auc": 0.9706358974192817,
  "anchor": 0.27402,
  "delta": -0.1495822703878245
}
                        mean_p  realized      n
bin                                            
(-0.001, 7.66e-05]    0.000069  0.000308  58442
(7.66e-05, 0.000348]  0.000335  0.000439  68298
(0.000348, 0.000357]  0.000357  0.000923  29257
(0.000357, 0.00123]   0.001028  0.000944  52970
(0.00123, 0.00264]    0.002226  0.001731  32348
(0.00264, 0.00773]    0.004797  0.003927  46857
(0.00773, 0.0219]     0.013504  0.012139  48520
(0.0219, 0.0908]      0.051518  0.049869  46522
(0.0908, 0.402]       0.218885  0.221510  47596
(0.402, 1.0]          0.743034  0.767708  46588
-------------- mnq-3STREAM-VEL-9-12am 3s --------------
    joblib_file: stage-5/model_raw_mnq-3STREAM-VEL-9-12am_3s.joblib
importance_file: stage-5/importance_raw_mnq-3STREAM-VEL-9-12am_3s.csv
      pred_file: stage-5/pred_raw_mnq-3STREAM-VEL-9-12am_3s.pqt
holdout-window logloss: 0.1243114173412323

--------------------------- SUMMARY ---------------------------

GREEN 0.0026400704  49.73% of bars  wrong 1/1365
RED   0.40172413    10.01% of bars  right 75.9%
y.mean 0.10386  ll_model 0.12443  ll_const 0.33348  skill 0.6269  rows 477398
'''

'\n--------------------- !! VERIFY !! ---------------------\nFRAME: 3sec, STAGE0_TAG: mnq-3STREAM-VEL-9-12am, BODY_TAG: raw\n----------------------- MANIFEST -----------------------\n{\n  "frame_seconds": 3,\n  "session_start": "09:00",\n  "session_end": "12:00",\n  "warmup_bars": 10,\n  "streams": [\n    {\n      "name": "MNQ_D1",\n      "column": "jmaD1"\n    },\n    {\n      "name": "MNQ_D2",\n      "column": "jmaD2"\n    },\n    {\n      "name": "MNQ_VEL",\n      "column": "VEL"\n    }\n  ],\n  "self_stream": "MNQ_JMA_SELF",\n  "source_file": "data/mnq-tick-full-3sec.pqt",\n  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",\n  "n_bars": 4120088,\n  "n_sessions": 1145,\n  "date_min": "2022-01-03 00:00:00",\n  "date_max": "2026-07-08 00:00:00",\n  "created": "2026-07-26T14:11:03.993239",\n  "stage0_tag": "mnq-3STREAM-VEL-9-12am"\n}\n--------------------------------------------------------\nwelford max abs diff: 4.440892098500626e-15\n[200]\tvalid\'s binary_logloss: 0.133575\n[400]\tvalid\'s

In [13]:
'''
--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: mnq-3STREAM-RSX-9-12am, BODY_TAG: raw
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "MNQ_RSX",
      "column": "RSX"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 4120088,
  "n_sessions": 1145,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-08 00:00:00",
  "created": "2026-07-26T14:19:31.911837",
  "stage0_tag": "mnq-3STREAM-RSX-9-12am"
}
--------------------------------------------------------
welford max abs diff: 4.440892098500626e-15
[200]	valid's binary_logloss: 0.133701
[400]	valid's binary_logloss: 0.129928
[600]	valid's binary_logloss: 0.128884
[800]	valid's binary_logloss: 0.128403
[1000]	valid's binary_logloss: 0.128089
[1200]	valid's binary_logloss: 0.127947
[1400]	valid's binary_logloss: 0.127889
[1600]	valid's binary_logloss: 0.127864
[1800]	valid's binary_logloss: 0.127835
[2000]	valid's binary_logloss: 0.127851
{
  "n_train": 3175513,
  "n_valid": 452137,
  "best_iteration": 1876,
  "valid_logloss_cal": 0.12737259057585823
}
                feature         gain
      z_body_raw_signed 2.060314e+06
            z_jmaD1_mag 1.561159e+06
         z_jmaD2_signed 1.355124e+06
 z_body_raw_signed_lag1 1.066232e+06
         z_jmaD1_signed 9.738057e+05
         z_body_raw_mag 9.426930e+05
       z_jmaD1_mag_lag1 5.809305e+05
    z_jmaD1_signed_lag1 3.498163e+05
    z_body_raw_mag_lag1 3.190108e+05
     MNQ_D1|conf|bsince 3.024566e+05
      MNQ_D1|opp|bsince 2.664703e+05
 z_body_raw_signed_lag2 2.528475e+05
            z_jmaD2_mag 2.113657e+05
        MNQ_D1|opp|ewm2 1.762147e+05
       z_jmaD1_mag_lag2 1.486775e+05
  MNQ_JMA_SELF|all|ewm2 1.476488e+05
MNQ_JMA_SELF|all|bsince 1.424962e+05
                    tod 1.195277e+05
    z_jmaD1_signed_lag2 1.071791e+05
    z_jmaD2_signed_lag1 1.007984e+05
       MNQ_D1|conf|ewm2 9.699714e+04
       z_jmaD2_mag_lag1 9.398279e+04
    z_body_raw_mag_lag2 6.913845e+04
       z_jmaD2_mag_lag2 6.573766e+04
        MNQ_D1|opp|ewm6 6.179391e+04
    z_jmaD2_signed_lag2 5.036891e+04
 MNQ_JMA_SELF|all|ewm18 4.853136e+04
        MNQ_D2|opp|ewm2 4.737058e+04
         z_leg_amp_lag1 4.531400e+04
              z_leg_amp 4.317453e+04
       MNQ_D1|opp|ewm18 4.214720e+04
      MNQ_D1|conf|ewm18 4.172734e+04
         z_leg_amp_lag2 4.168034e+04
       MNQ_D2|opp|ewm18 4.013819e+04
       MNQ_D2|conf|ewm2 3.984282e+04
      MNQ_D2|conf|ewm18 3.955693e+04
     MNQ_RSX|conf|ewm18 3.890578e+04
              z_RSX_mag 3.414541e+04
        MNQ_D2|opp|ewm6 3.325268e+04
  MNQ_JMA_SELF|all|ewm6 3.219297e+04
      MNQ_RSX|opp|ewm18 3.150511e+04
         z_RSX_mag_lag2 3.050094e+04
           z_RSX_signed 3.036254e+04
                    age 2.900923e+04
       MNQ_D2|conf|ewm6 2.741346e+04
      MNQ_RSX|conf|ewm2 2.734813e+04
      z_RSX_signed_lag2 2.685132e+04
       MNQ_D1|conf|ewm6 2.658408e+04
      MNQ_RSX|conf|ewm6 2.651258e+04
         z_RSX_mag_lag1 2.480723e+04
      z_RSX_signed_lag1 2.452655e+04
     MNQ_RSX|opp|bsince 1.995557e+04
       MNQ_RSX|opp|ewm6 1.950174e+04
       MNQ_RSX|opp|ewm2 1.655954e+04
     MNQ_D2|conf|bsince 8.867351e+03
    MNQ_RSX|conf|bsince 5.265454e+03
      MNQ_D2|opp|bsince 3.300975e+03
{
  "n_rows": 477398,
  "holdout_logloss_cal": 0.12436663540827801,
  "holdout_logloss_const": 0.3334844163241089,
  "skill": 0.6270691243113208,
  "auc": 0.9705495974618894,
  "anchor": 0.27402,
  "delta": -0.14965336459172196
}
                        mean_p  realized      n
bin                                            
(-0.001, 8.74e-05]    0.000081  0.000322  62199
(8.74e-05, 0.000381]  0.000360  0.000478  62755
(0.000381, 0.000563]  0.000538  0.000825  46071
(0.000563, 0.00124]   0.000916  0.001126  21318
(0.00124, 0.00264]    0.002177  0.001462  48549
(0.00264, 0.00716]    0.005019  0.004185  55201
(0.00716, 0.0229]     0.015597  0.015537  45376
(0.0229, 0.0867]      0.053809  0.049952  40479
(0.0867, 0.407]       0.223314  0.227385  50773
(0.407, 1.0]          0.758649  0.781095  44677
-------------- mnq-3STREAM-RSX-9-12am 3s --------------
    joblib_file: stage-5/model_raw_mnq-3STREAM-RSX-9-12am_3s.joblib
importance_file: stage-5/importance_raw_mnq-3STREAM-RSX-9-12am_3s.csv
      pred_file: stage-5/pred_raw_mnq-3STREAM-RSX-9-12am_3s.pqt
holdout-window logloss: 0.1243666484951973

--------------------------- SUMMARY ---------------------------

GREEN 0.0026417852  46.44% of bars  wrong 1/1449
RED   0.40719259    10.11% of bars  right 75.6%
y.mean 0.10386  ll_model 0.12437  ll_const 0.33348  skill 0.6271  rows 477398
'''

'\n--------------------- !! VERIFY !! ---------------------\nFRAME: 3sec, STAGE0_TAG: mnq-3STREAM-RSX-9-12am, BODY_TAG: raw\n----------------------- MANIFEST -----------------------\n{\n  "frame_seconds": 3,\n  "session_start": "09:00",\n  "session_end": "12:00",\n  "warmup_bars": 10,\n  "streams": [\n    {\n      "name": "MNQ_D1",\n      "column": "jmaD1"\n    },\n    {\n      "name": "MNQ_D2",\n      "column": "jmaD2"\n    },\n    {\n      "name": "MNQ_RSX",\n      "column": "RSX"\n    }\n  ],\n  "self_stream": "MNQ_JMA_SELF",\n  "source_file": "data/mnq-tick-full-3sec.pqt",\n  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",\n  "n_bars": 4120088,\n  "n_sessions": 1145,\n  "date_min": "2022-01-03 00:00:00",\n  "date_max": "2026-07-08 00:00:00",\n  "created": "2026-07-26T14:19:31.911837",\n  "stage0_tag": "mnq-3STREAM-RSX-9-12am"\n}\n--------------------------------------------------------\nwelford max abs diff: 4.440892098500626e-15\n[200]\tvalid\'s binary_logloss: 0.133701\n[400]\tvalid\'s

In [14]:
'''
--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: mnq-4STREAM-RSX-9-12am, BODY_TAG: raw
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    },
    {
      "name": "TICK_D1",
      "column": "tickJmaD1"
    },
    {
      "name": "MNQ_RSX",
      "column": "RSX"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 4120088,
  "n_sessions": 1145,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-08 00:00:00",
  "created": "2026-07-26T14:24:50.023439",
  "stage0_tag": "mnq-4STREAM-RSX-9-12am"
}
--------------------------------------------------------
welford max abs diff: 4.440892098500626e-15
[200]	valid's binary_logloss: 0.133347
[400]	valid's binary_logloss: 0.129087
[600]	valid's binary_logloss: 0.127824
[800]	valid's binary_logloss: 0.127333
[1000]	valid's binary_logloss: 0.127077
[1200]	valid's binary_logloss: 0.126928
[1400]	valid's binary_logloss: 0.126855
[1600]	valid's binary_logloss: 0.126808
[1800]	valid's binary_logloss: 0.126805
{
  "n_train": 3175513,
  "n_valid": 452137,
  "best_iteration": 1699,
  "valid_logloss_cal": 0.12632095600769994
}
                feature         gain
      z_body_raw_signed 2.033477e+06
            z_jmaD1_mag 1.393147e+06
         z_jmaD2_signed 1.294807e+06
         z_jmaD1_signed 1.082972e+06
 z_body_raw_signed_lag1 1.067788e+06
         z_body_raw_mag 9.645534e+05
       z_jmaD1_mag_lag1 5.225091e+05
    z_jmaD1_signed_lag1 3.384841e+05
    z_body_raw_mag_lag1 3.301439e+05
      MNQ_D1|opp|bsince 2.968965e+05
     MNQ_D1|conf|bsince 2.780508e+05
 z_body_raw_signed_lag2 2.613446e+05
MNQ_JMA_SELF|all|bsince 2.365177e+05
            z_jmaD2_mag 2.225542e+05
       z_jmaD1_mag_lag2 1.867715e+05
        MNQ_D1|opp|ewm2 1.501896e+05
                    tod 1.179549e+05
  MNQ_JMA_SELF|all|ewm2 1.155190e+05
    z_jmaD1_signed_lag2 1.113837e+05
    z_jmaD2_signed_lag1 9.385067e+04
{
  "n_rows": 477398,
  "holdout_logloss_cal": 0.12291682276219111,
  "holdout_logloss_const": 0.3334844163241089,
  "skill": 0.6314165917644261,
  "auc": 0.9716632444517268,
  "anchor": 0.27402,
  "delta": -0.15110317723780886
}
                        mean_p  realized      n
bin                                            
(-0.001, 8.15e-05]    0.000066  0.000113  61781
(8.15e-05, 0.000304]  0.000233  0.000419  66894
(0.000304, 0.000313]  0.000310  0.000342  20478
(0.000313, 0.00113]   0.000907  0.000855  61981
(0.00113, 0.00206]    0.001919  0.001842  29862
(0.00206, 0.00583]    0.004112  0.003335  45584
(0.00583, 0.0264]     0.013585  0.012188  52266
(0.0264, 0.0929]      0.052153  0.050329  44229
(0.0929, 0.396]       0.220579  0.223743  48283
(0.396, 1.0]          0.750962  0.773566  46040
-------------- mnq-4STREAM-RSX-9-12am 3s --------------
    joblib_file: stage-5/model_raw_mnq-4STREAM-RSX-9-12am_3s.joblib
importance_file: stage-5/importance_raw_mnq-4STREAM-RSX-9-12am_3s.csv
      pred_file: stage-5/pred_raw_mnq-4STREAM-RSX-9-12am_3s.pqt
holdout-window logloss: 0.12279051542282104

--------------------------- SUMMARY ---------------------------

GREEN 0.0020554066  47.93% of bars  wrong 1/1816
RED   0.39620537    10.03% of bars  right 76.0%
y.mean 0.10386  ll_model 0.12291  ll_const 0.33348  skill 0.6314  rows 477398
'''

'\n--------------------- !! VERIFY !! ---------------------\nFRAME: 3sec, STAGE0_TAG: mnq-4STREAM-RSX-9-12am, BODY_TAG: raw\n----------------------- MANIFEST -----------------------\n{\n  "frame_seconds": 3,\n  "session_start": "09:00",\n  "session_end": "12:00",\n  "warmup_bars": 10,\n  "streams": [\n    {\n      "name": "MNQ_D1",\n      "column": "jmaD1"\n    },\n    {\n      "name": "MNQ_D2",\n      "column": "jmaD2"\n    },\n    {\n      "name": "TICK_D1",\n      "column": "tickJmaD1"\n    },\n    {\n      "name": "MNQ_RSX",\n      "column": "RSX"\n    }\n  ],\n  "self_stream": "MNQ_JMA_SELF",\n  "source_file": "data/mnq-tick-full-3sec.pqt",\n  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",\n  "n_bars": 4120088,\n  "n_sessions": 1145,\n  "date_min": "2022-01-03 00:00:00",\n  "date_max": "2026-07-08 00:00:00",\n  "created": "2026-07-26T14:24:50.023439",\n  "stage0_tag": "mnq-4STREAM-RSX-9-12am"\n}\n--------------------------------------------------------\nwelford max abs diff: 4.44089209

In [15]:
'''
--------------------- !! VERIFY !! ---------------------
FRAME: 3sec, STAGE0_TAG: mnq-2STREAM-9-12am, BODY_TAG: raw
----------------------- MANIFEST -----------------------
{
  "frame_seconds": 3,
  "session_start": "09:00",
  "session_end": "12:00",
  "warmup_bars": 10,
  "streams": [
    {
      "name": "MNQ_D1",
      "column": "jmaD1"
    },
    {
      "name": "MNQ_D2",
      "column": "jmaD2"
    }
  ],
  "self_stream": "MNQ_JMA_SELF",
  "source_file": "data/mnq-tick-full-3sec.pqt",
  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",
  "n_bars": 4120088,
  "n_sessions": 1145,
  "date_min": "2022-01-03 00:00:00",
  "date_max": "2026-07-08 00:00:00",
  "created": "2026-07-26T14:38:32.474391",
  "stage0_tag": "mnq-2STREAM-9-12am"
}
--------------------------------------------------------
welford max abs diff: 4.440892098500626e-15
[200]	valid's binary_logloss: 0.133656
[400]	valid's binary_logloss: 0.129867
[600]	valid's binary_logloss: 0.128805
[800]	valid's binary_logloss: 0.128352
[1000]	valid's binary_logloss: 0.128108
[1200]	valid's binary_logloss: 0.127958
[1400]	valid's binary_logloss: 0.127881
[1600]	valid's binary_logloss: 0.127847
[1800]	valid's binary_logloss: 0.127837
{
  "n_train": 3175513,
  "n_valid": 452137,
  "best_iteration": 1768,
  "valid_logloss_cal": 0.12738557770625883
}
                feature         gain
      z_body_raw_signed 2.040219e+06
            z_jmaD1_mag 1.535917e+06
         z_jmaD2_signed 1.369007e+06
 z_body_raw_signed_lag1 1.066134e+06
         z_body_raw_mag 9.925696e+05
         z_jmaD1_signed 9.780656e+05
       z_jmaD1_mag_lag1 5.693957e+05
    z_body_raw_mag_lag1 3.593931e+05
    z_jmaD1_signed_lag1 3.156301e+05
      MNQ_D1|opp|bsince 2.988211e+05
     MNQ_D1|conf|bsince 2.985817e+05
 z_body_raw_signed_lag2 2.655607e+05
            z_jmaD2_mag 2.283408e+05
        MNQ_D1|opp|ewm2 1.765192e+05
       z_jmaD1_mag_lag2 1.633233e+05
  MNQ_JMA_SELF|all|ewm2 1.605897e+05
MNQ_JMA_SELF|all|bsince 1.451373e+05
    z_jmaD1_signed_lag2 1.212089e+05
                    tod 1.200016e+05
    z_jmaD2_signed_lag1 1.054851e+05
{
  "n_rows": 477398,
  "holdout_logloss_cal": 0.12431210511687651,
  "holdout_logloss_const": 0.3334844163241089,
  "skill": 0.6272326410717217,
  "auc": 0.9706181574008637,
  "anchor": 0.27402,
  "delta": -0.14970789488312347
}
                       mean_p  realized      n
bin                                           
(-0.001, 8.9e-05]    0.000057  0.000298  53612
(8.9e-05, 0.00037]   0.000325  0.000408  53907
(0.00037, 0.000558]  0.000528  0.000708  67773
(0.000558, 0.00111]  0.001078  0.001361  27176
(0.00111, 0.00246]   0.002114  0.001822  43355
(0.00246, 0.00635]   0.004961  0.003723  42175
(0.00635, 0.0227]    0.014403  0.013337  52636
(0.0227, 0.0934]     0.053118  0.051842  41935
(0.0934, 0.402]      0.223697  0.227640  49956
(0.402, 1.0]         0.757304  0.779444  44873
-------------- mnq-2STREAM-9-12am 3s --------------
    joblib_file: stage-5/model_raw_mnq-2STREAM-9-12am_3s.joblib
importance_file: stage-5/importance_raw_mnq-2STREAM-9-12am_3s.csv
      pred_file: stage-5/pred_raw_mnq-2STREAM-9-12am_3s.pqt
holdout-window logloss: 0.124269999563694

--------------------------- SUMMARY ---------------------------

GREEN 0.0024648116  47.96% of bars  wrong 1/1422
RED   0.40214646    10.09% of bars  right 75.5%
y.mean 0.10386  ll_model 0.12431  ll_const 0.33348  skill 0.6272  rows 477398
'''

'\n--------------------- !! VERIFY !! ---------------------\nFRAME: 3sec, STAGE0_TAG: mnq-2STREAM-9-12am, BODY_TAG: raw\n----------------------- MANIFEST -----------------------\n{\n  "frame_seconds": 3,\n  "session_start": "09:00",\n  "session_end": "12:00",\n  "warmup_bars": 10,\n  "streams": [\n    {\n      "name": "MNQ_D1",\n      "column": "jmaD1"\n    },\n    {\n      "name": "MNQ_D2",\n      "column": "jmaD2"\n    }\n  ],\n  "self_stream": "MNQ_JMA_SELF",\n  "source_file": "data/mnq-tick-full-3sec.pqt",\n  "raw_file": "data/mnq-ohlc-raw-3sec.pqt",\n  "n_bars": 4120088,\n  "n_sessions": 1145,\n  "date_min": "2022-01-03 00:00:00",\n  "date_max": "2026-07-08 00:00:00",\n  "created": "2026-07-26T14:38:32.474391",\n  "stage0_tag": "mnq-2STREAM-9-12am"\n}\n--------------------------------------------------------\nwelford max abs diff: 4.440892098500626e-15\n[200]\tvalid\'s binary_logloss: 0.133656\n[400]\tvalid\'s binary_logloss: 0.129867\n[600]\tvalid\'s binary_logloss: 0.128805\n[80


| streams | holdout LL | skill | AUC | features |
|---|---|---|---|---|
| 2 (MNQ only) | 0.124312 | 0.627233 | 0.970618 | 43 |
| **3 (+TICK_D1)** | **0.122963** | **0.63128** | **0.971615** | **57** |
| 4 (+TICK_D2) | 0.123117 | 0.63082 | 0.971575 | 71 |
| 5 (+RSX) | 0.123156 | 0.63070 | 0.971548 | 85 |
| 4 (+RSX) | 0.122917 | 0.63142 | 0.971615 | ?? |
| 4 (+VEL) | 0.126193 | 0.63132 | 0.971685 | ?? |
| 3 (+VEL) | 0.124438 | 0.62686 | 0.970636 | ?? |
| 3 (+RSX) | 0.124367 | 0.62707 | 0.970550 | ?? |

  "holdout_logloss_cal": 0.12431210511687651,
  "skill": 0.6272326410717217,
  "auc": 0.9706181574008637,

  example:
  
  "holdout_logloss_cal": 0.12296254907057177, -> holdout LL
  "skill": 0.6312794749873225,                -> skill
  "auc": 0.9716153733084922,                  -> auc

  
That's coherent and it finally resolves the TICK question properly: **TICK_D1 is worth about +0.0013; TICK_D2 costs about −0.00015.** The earlier no-TICK test dropped both at once, so TICK_D2's drag was hidden inside TICK_D1's contribution. Splitting them separates the two, and the shape is exactly what the Stage-0 diagnostics implied — TICK_D2 fires 25% more often than TICK_D1 with nearly identical recall and lower precision at every lag. Redundant, faster-firing, slightly noisier.

Two honest caveats on the size of the win. The +0.00015 improvement over 4 streams is small enough to sit near LightGBM's own run-to-run variation (bagging at 0.8, feature_fraction 0.9), so treat "3 beats 4" as *at worst a tie*, not a proven gain. What's solid is the direction across all four rows: every subtraction of a redundant stream is neutral-or-better, every addition is neutral-or-worse. That's the data-saturation signature again, now measured four ways.

Calibration is unchanged in character — top bin 0.753 predicted / 0.776 realized, same mild safe-side under-confidence you've had throughout.

**On whether to redeploy:** not for the 0.00015. Changing the production model costs new decile thresholds in the SC study, a new monitor baseline, and a golden-replay re-run — real work and error surface for a gain you can't measure live. But the *simplification* travels with it for free if you're redeploying anyway: wire narrows from `<i7f>` 32B to `<i6f>` 28B, 71 → 57 features, one less stream to keep alive in SC. So: keep running what's running; make 3-stream the config for the next scheduled refresh, whenever data extension or a monitor ALERT prompts one.

If you do adopt it, pull the thresholds from this model's own summary block — `p_cal.quantile(0.50)` and `.quantile(0.90)` — rather than reading them off the decile table, since the qcut bins here are uneven (ties in the low tail) and the 50% cut sits inside the fifth bin rather than on its edge.

Stream count is now settled empirically at 3, and with wicks, TICK_D2, and RSX all returning null-or-negative, the feature side of this model is closed. The next real gain has to come from a different data source, not another indicator on the same bars.